In [2]:
# =========================
# 1. Install Required Libraries
# =========================
!pip install -q kaggle xgboost

# =========================
# 2. Upload kaggle.json
# =========================
from google.colab import files
files.upload()

# =========================
# 3. Setup Kaggle API
# =========================
import os
import zipfile

os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# =========================
# 4. Download Dataset
# =========================
!kaggle datasets download -d naiyakhalid/flood-prediction-dataset

# =========================
# 5. Extract Dataset
# =========================
with zipfile.ZipFile("flood-prediction-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

# =========================
# 6. Load Dataset
# =========================
import pandas as pd

df = pd.read_csv("data/flood.csv")   # Adjust name if needed
df.head()


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/naiyakhalid/flood-prediction-dataset
License(s): CC0-1.0
100% 28.6M/28.6M [00:00<00:00, 115MB/s] 



,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,Encroachments,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,3,8,6,6,4,4,6,2,3,2,...,10,7,4,2,3,4,3,2,6,0.450
1,8,4,5,7,7,9,1,5,5,4,...,9,2,6,2,1,1,9,1,3,0.475
2,3,10,4,1,7,5,4,7,4,9,...,7,4,4,8,6,1,8,3,6,0.515
3,4,4,2,7,3,4,1,4,6,4,...,4,2,6,6,8,8,6,6,10,0.520
4,3,7,5,2,5,8,5,2,7,5,...,7,6,5,3,3,4,4,3,4,0.475


In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# =========================
# 1. Load Train and Test
# =========================
train_df = pd.read_csv("data/train.csv")

print("TRAIN DATA")
display(train_df.head())

TRAIN DATA


,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,0,5,8,5,8,6,4,4,3,3,...,5,3,3,5,4,7,5,7,3,0.445
1,1,6,7,4,4,8,8,3,5,4,...,7,2,0,3,5,3,3,4,3,0.450
2,2,6,5,6,7,3,7,1,5,4,...,7,3,7,5,6,8,2,3,3,0.530
3,3,3,4,6,5,4,8,4,7,6,...,2,4,7,4,4,6,5,7,5,0.535
4,4,5,3,2,6,4,4,3,3,3,...,2,2,6,6,4,1,2,3,5,0.415


In [4]:
len(train_df)

1117957

In [5]:
test_df = pd.read_csv("data/flood.csv")

# =========================
# 2. Drop ID Column (if exists)
# =========================
if "id" in train_df.columns:
    train_df = train_df.drop("id", axis=1)

if "id" in test_df.columns:
    test_df = test_df.drop("id", axis=1)

# =========================
# 3. Separate Features & Target
# =========================
X_train = train_df.drop("FloodProbability", axis=1)
y_train = train_df["FloodProbability"]

X_test = test_df.drop("FloodProbability", axis=1)
y_test = test_df["FloodProbability"]

# =========================
# 4. Train Model
# =========================
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# =========================
# 5. Predict
# =========================
y_pred = model.predict(X_test)

# =========================
# 6. Evaluate
# =========================
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Flood.csv RMSE:", rmse)
print("Flood.csv R2 Score:", r2)


Flood.csv RMSE: 0.011875352596356968
Flood.csv R2 Score: 0.9436668032384458


In [6]:
# =========================
# 7. Save Model (XGBoost Native)
# =========================
model.save_model("flood_model.json")

print("Model saved as flood_model.json")

Model saved as flood_model.json


In [7]:
# =========================
# 7. Save Model (Joblib)
# =========================
import joblib

joblib.dump(model, "flood_model.pkl")

print("Model saved as flood_model.pkl")

Model saved as flood_model.pkl


In [8]:
from xgboost import XGBRegressor
import numpy as np

# Load model
model = XGBRegressor()
model.load_model("flood_model.json")

def predict(data):
    data = np.array(data)
    prediction = model.predict(data)
    return prediction.tolist()

In [10]:
# =========================
# 7. Auto Generate HF Files
# =========================
import json
import pkg_resources

# -------- requirements.txt --------
required_packages = ["xgboost", "numpy"]

installed_packages = {pkg.key: pkg.version for pkg in pkg_resources.working_set}

with open("requirements.txt", "w") as f:
    for pkg in required_packages:
        if pkg in installed_packages:
            f.write(f"{pkg}=={installed_packages[pkg]}\n")
        else:
            f.write(f"{pkg}\n")

print("requirements.txt generated")

# -------- config.json --------
config = {
    "model_type": "xgboost",
    "framework": "sklearn",
    "task": "regression"
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json generated")

/tmp/ipykernel_3546/3843935334.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


requirements.txt generated
config.json generated


In [11]:
# =========================
# 8. Zip Everything
# =========================
import zipfile

with zipfile.ZipFile("flood_model_hf.zip", "w") as zipf:
    zipf.write("flood_model.json")
    zipf.write("requirements.txt")
    zipf.write("config.json")

print("ZIP file ready for upload")

ZIP file ready for upload


In [15]:
# =========================
# 7. Full Evaluation Metrics
# =========================
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
    explained_variance_score
)
import numpy as np

# Basic Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Advanced Metrics
mape = mean_absolute_percentage_error(y_test, y_pred)
explained_var = explained_variance_score(y_test, y_pred)

# Custom Metrics
max_error = np.max(np.abs(y_test - y_pred))

# Accuracy-like metric (for regression, tolerance-based)
tolerance = 0.1  # adjust if needed
accuracy = np.mean(np.abs(y_test - y_pred) < tolerance)

# =========================
# Print Results
# =========================
print("========== MODEL EVALUATION ==========")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.4f}")
print(f"R2 Score: {r2:.4f}")
print(f"Explained Variance: {explained_var:.4f}")
print(f"Max Error: {max_error:.4f}")
residuals = y_test - y_pred

print("\nResidual Stats:")
print(f"Mean Residual: {np.mean(residuals):.4f}")
print(f"Std Residual: {np.std(residuals):.4f}")


========== MODEL EVALUATION ==========
MSE: 0.0001
RMSE: 0.0119
MAE: 0.0107
MAPE: 0.0212
R2 Score: 0.9437
Explained Variance: 0.9883
Max Error: 0.0381

Residual Stats:
Mean Residual: -0.0106
Std Residual: 0.0054
